# Calibración de la cámara
Calibración de la webcam del portátil con el patrón de ajedrez impreso (10x7 cuadrados de 25 mm). Se hace una vez por cámara, antes de ejecutar el sistema.

## 0. Preparación

In [1]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


## 1. Carga de imágenes
Imágenes capturadas con `capture_images.py`. Cada webcam se calibra por separado.

In [2]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

In [3]:
imgs_path = sorted(glob.glob('../data/calibration/images/*.png'))
imgs = load_images(imgs_path)

print(f"Number of images: {len(imgs)}")
print(f"First image shape: {imgs[0].shape}")

Number of images: 25
First image shape: (480, 640, 3)


## 2. Detección de esquinas

In [4]:
# The printed pattern has 10x7 squares -> 9x6 inner corners
chessboard_shape = (9, 6)
corners = [cv2.findChessboardCorners(img, chessboard_shape) for img in imgs]

In [5]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

# cv2.cornerSubPix() refines the array IN PLACE, so we refine the copy
corners_refined = []
for gray, (found, c) in zip(imgs_gray, corners_copy):
    if found:
        c = cv2.cornerSubPix(gray, c, (5, 5), (-1, -1), criteria)
    corners_refined.append((found, c))

In [6]:
imgs_refined = [
    cv2.drawChessboardCorners(img, chessboard_shape, c, found)
    for img, (found, c) in zip(copy.deepcopy(imgs), corners_refined)
]

In [7]:
def show_image(img: np.array, img_name: str = "Image"):
    cv2.imshow(img_name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def write_image(output_folder: str, img_name: str, img: np.array):
    os.makedirs(output_folder, exist_ok=True)
    img_path = os.path.join(output_folder, img_name)
    cv2.imwrite(img_path, img)

In [8]:
output_folder = "../output/calibration"

show_image(imgs_refined[0], "Refined detection")
for i, img in enumerate(imgs_refined):
    write_image(output_folder, f"corners_{i:03d}.png", img)

## 3. Puntos del patrón

In [9]:
def get_chessboard_points(chessboard_shape, dx, dy):
    n_cols, n_rows = chessboard_shape
    points = np.zeros((n_rows * n_cols, 3), np.float32)   # z = 0 because the pattern is planar
    grid = np.mgrid[0:n_cols, 0:n_rows].T.reshape(-1, 2)
    points[:, :2] = grid * np.array([dx, dy], dtype=np.float32)
    return points

In [10]:
square_size = 25    # side of every square of the printed pattern [mm] (measure it!)
chessboard_points = [get_chessboard_points(chessboard_shape, square_size, square_size) for _ in imgs]

## 4. Calibración

In [11]:
valid_corners = [c for found, c in corners_refined if found]
valid_points = [p for p, (found, _) in zip(chessboard_points, corners_refined) if found]
valid_corners = np.array(valid_corners, dtype=np.float32)
valid_points = np.array(valid_points, dtype=np.float32)

print("Valid views:", len(valid_corners), "/", len(imgs))

Valid views: 25 / 25


In [12]:
image_size = imgs_gray[0].shape[::-1]     # (width, height)

rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(list(valid_points), list(valid_corners), image_size, None, None)

print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[583.88039227   0.         322.92294775]
 [  0.         583.48622355 251.54966312]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[-0.05620641  0.33649578  0.00070961  0.00167201 -0.44379562]]
Root mean squared reprojection error:
 0.47540581509583635


## 5. Guardar los parámetros
El programa principal carga este archivo al arrancar.

In [13]:
np.savez("../data/calibration/params.npz",
         intrinsics=intrinsics, dist_coeffs=dist_coeffs, rms=rms, image_size=image_size)

## 6. Corrección de la distorsión

In [14]:
# Same K as new camera matrix and no rectification (R = None)
map1, map2 = cv2.initUndistortRectifyMap(intrinsics, dist_coeffs, None, intrinsics, image_size, cv2.CV_16SC2)

undistorted_imgs = [cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
                    for img in imgs[:2]]

for i, (original, undistorted) in enumerate(zip(imgs[:2], undistorted_imgs)):
    comparison = np.hstack((original, undistorted))     # original (left) vs undistorted (right)
    show_image(comparison, f"Image {i}: original | undistorted")
    write_image(output_folder, f"comparison_{i:03d}.png", comparison)

## 7. Comprobación: ¿la corrección mejora la imagen?
Las esquinas de cada fila del tablero están en una recta en la realidad. Medimos cuánto se separan de una recta antes y después de corregir la distorsión (distancia media en píxeles).

In [15]:
def line_error(points):
    # Fit a straight line to the points and return the mean distance to it [px]
    vx, vy, x0, y0 = cv2.fitLine(points, cv2.DIST_L2, 0, 0.01, 0.01).ravel()
    distances = np.abs((points[:, 0] - x0) * vy - (points[:, 1] - y0) * vx)
    return distances.mean()

n_cols, n_rows = chessboard_shape
errors_before, errors_after = [], []
for c in valid_corners:
    c_undistorted = cv2.undistortPoints(c, intrinsics, dist_coeffs, P=intrinsics)
    for r in range(n_rows):
        row = slice(r * n_cols, (r + 1) * n_cols)
        errors_before.append(line_error(c.reshape(-1, 2)[row]))
        errors_after.append(line_error(c_undistorted.reshape(-1, 2)[row]))

print(f"Mean distance to a straight line BEFORE: {np.mean(errors_before):.3f} px")
print(f"Mean distance to a straight line AFTER:  {np.mean(errors_after):.3f} px")

Mean distance to a straight line BEFORE: 0.146 px
Mean distance to a straight line AFTER:  0.144 px
